# Volume Prediction with XGBoost (Gradient Boosting Baseline)

Identical preprocessing pipeline as `mhf-volume.ipynb` (DeepLOB), but replaces the neural network
with XGBoost gradient boosting.

**Purpose:** Determine whether volume prediction difficulty is due to model architecture
(DeepLOB not suited for volume) or the fundamental unpredictability of per-second volume.

**Setup:** Same log1p transform, same z-score normalization, same chrono split, same features.
Only the model changes: 168 hand-crafted summary features fed into 60 independent XGBoost regressors.

In [ ]:
import os, sys, subprocess

# Get project files onto remote runtime (Colab extension doesn't sync files)
if not os.path.isdir("/content/Ultramarin/utils"):
    subprocess.run(["git", "clone", "-q", "-b", "potentially-improve-model",
                    "https://github.com/JosephZacharyGawlik/Ultramarin.git"],
                   cwd="/content")
os.chdir("/content/Ultramarin")
sys.path.insert(0, os.getcwd())

import numpy as np
from pathlib import Path
import pandas as pd
import polars as pl
import torch
import random
from utils.utils import compute_ofi
from data.simulate_walk_the_book import simulate_walk_the_book
from utils.datastuff import TrainCfg, LOBProcessor
from utils.train import chrono_split
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
import warnings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Copy data from Google Drive (data/ is gitignored, parquets don't sync)
import os, subprocess

dst = "/content/Ultramarin/data"

if not os.path.isfile(os.path.join(dst, "BTCUSDT/X_train.parquet")):
    print("Data not found. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')

    src = "/content/drive/MyDrive/data"
    os.makedirs(dst, exist_ok=True)
    print(f"Copying from {src} to {dst} ...")

    result = subprocess.run(["cp", "-r", f"{src}/.", dst], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"ERROR copying data: {result.stderr}")
    else:
        print("Data copied successfully.")

    print(f"data/ contents: {os.listdir(dst)}")
    btc_dir = os.path.join(dst, "BTCUSDT")
    if os.path.isdir(btc_dir):
        print(f"data/BTCUSDT/ contents: {os.listdir(btc_dir)}")
else:
    print("Data already present.")

In [ ]:
# Paths and volume_to_fill
from pathlib import Path
import re

root = Path("data") if Path("data").exists() else Path.cwd()

data_asset = "BTCUSDT"
symbol_dir = root / data_asset
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
X_test_path = symbol_dir / "X_test.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

# Optimal K per currency (from all-currency K-sweep on full training data)
OPTIMAL_K = {
    "BTCUSDT": 14, "ETHUSDT": 26, "LTCUSDT": 16,
    "SOLUSDT": 17, "ADAUSDT": 7, "DOGEUSDT": 20, "XRPUSDT": 20,
}
K_SECONDS = OPTIMAL_K.get(data_asset, 14)

# Known volumes per currency (fallback if vol_to_fill.txt is missing)
KNOWN_VOLUMES = {
    "BTCUSDT": 4.0, "ETHUSDT": 55.0, "LTCUSDT": 51.0,
    "SOLUSDT": 315.0, "ADAUSDT": 10436.0, "DOGEUSDT": 60349.0, "XRPUSDT": 13736.0,
}

volume_to_fill = None
if vol_path.exists():
    m = re.search(r"([\d.]+)", vol_path.read_text())
    if m:
        volume_to_fill = float(m.group(1))

if volume_to_fill is None:
    volume_to_fill = KNOWN_VOLUMES.get(data_asset)
    print(f"vol_to_fill.txt not found, using known volume: {volume_to_fill}")
else:
    print(f"volume_to_fill: {volume_to_fill}")

print(f"K_SECONDS: {K_SECONDS} (optimal for {data_asset})")

In [ ]:
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.append(f"ask_price_{i}")
    LOB_COLS.append(f"ask_vol_{i}")
    LOB_COLS.append(f"bid_price_{i}")
    LOB_COLS.append(f"bid_vol_{i}")

OFI_COLS = ['ofi_1', 'ofi_2', 'ofi_3', 'ofi_4', 'ofi_5', 'ofi_agg']

FEATURE_COLS = LOB_COLS + ["mid_price"] + OFI_COLS + ["volume"]

print(f"LOB Features: {len(LOB_COLS)}")
print(f"Extra Features: mid_price + {len(OFI_COLS)} OFI + volume = {1 + len(OFI_COLS) + 1}")
print(f"Total Features: {len(FEATURE_COLS)} (20 LOB + 1 mid_price + 6 OFI + 1 volume)")

In [ ]:
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# Data Preprocessing (identical to mhf-volume.ipynb)

Same pipeline: load parquets, chrono split, add mid_price, log1p volume, compute OFI,
LOBProcessor z-score normalization, feature reordering, volume target extraction.

In [ ]:
config = TrainCfg(

    # Hyperparameters (not all used by XGBoost, kept for LOBProcessor compatibility)
    epochs = 100,
    batch_size = 32,
    lr = 1e-3,
    weight_decay = 1e-5,
    smooth_lambda = 0.0,
    direction_lambda = 0.0,
    dropout = 0.1,
    tf_floor = 0.1,
    patience = 10,
    min_lr = 1e-6,

    # Windows
    input_window = 600,
    target_window = 60,
    val_ratio = 0.2,

    # Environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),

    # Features & Data
    x_path = X_path,
    y_path = Y_path,
    x_test_path = X_test_path,
    feature_cols = FEATURE_COLS,
)

# --- 1. Load Data ---
X_raw = pd.read_parquet(config.x_path).sort_values(["anonymized_id", "time_in_hour"])
Y_raw = pd.read_parquet(config.y_path).sort_values(["anonymized_id", "time_in_hour"])

# --- 2. Chrono Split ---
x_train_df, x_val_df, y_train_df, y_val_df = chrono_split(X_raw, Y_raw, val_ratio=config.val_ratio)

# --- 3. Add mid_price ---
x_train_df = x_train_df.assign(mid_price=(x_train_df["ask_price_1"] + x_train_df["bid_price_1"]) / 2.0)
x_val_df   = x_val_df.assign(mid_price=(x_val_df["ask_price_1"]     + x_val_df["bid_price_1"])   / 2.0)
y_train_df = y_train_df.assign(mid_price=(y_train_df["ask_price_1"] + y_train_df["bid_price_1"]) / 2.0)
y_val_df   = y_val_df.assign(mid_price=(y_val_df["ask_price_1"]     + y_val_df["bid_price_1"])   / 2.0)

# --- Log-transform volume before normalization ---
x_train_df = x_train_df.assign(volume=np.log1p(x_train_df["volume"].clip(lower=0)))
x_val_df   = x_val_df.assign(volume=np.log1p(x_val_df["volume"].clip(lower=0)))
y_train_df = y_train_df.assign(volume=np.log1p(y_train_df["volume"].clip(lower=0)))
y_val_df   = y_val_df.assign(volume=np.log1p(y_val_df["volume"].clip(lower=0)))
print("Applied log1p transform to volume column.")

# --- 4. Compute OFI features ---
x_train_df = compute_ofi(x_train_df)
x_val_df   = compute_ofi(x_val_df)
y_train_df = compute_ofi(y_train_df)
y_val_df   = compute_ofi(y_val_df)

# --- 5. Process with LOBProcessor ---
processor = LOBProcessor(config, device=config.device)

train_out = processor.process(pl.from_pandas(x_train_df), pl.from_pandas(y_train_df))
X_train_tensor, Y_train_tensor = train_out["X"], train_out["y"]

val_out = processor.process(pl.from_pandas(x_val_df), pl.from_pandas(y_val_df))
X_val_tensor, Y_val_tensor, val_id_map = val_out["X"], val_out["y"], val_out["y_id_map"]

# --- 6. Enforce FEATURE_COLS order and extract scalers ---
feature_indices = [processor.feature_map[col] for col in config.feature_cols]

X_train_tensor = X_train_tensor[:, :, feature_indices]
X_val_tensor   = X_val_tensor[:, :, feature_indices]
train_means    = train_out["means"][:, :, feature_indices]
train_stds     = train_out["stds"][:, :, feature_indices]

feature_index_map = {col: i for i, col in enumerate(config.feature_cols)}

# --- 7. Extract VOLUME as target ---
volume_idx = feature_index_map["volume"]
Y_train_tensor = Y_train_tensor[:, :, feature_indices]
Y_val_tensor   = Y_val_tensor[:, :, feature_indices]

vol_train = Y_train_tensor[:, :, volume_idx]  # [60, num_train_ids]
vol_val   = Y_val_tensor[:, :, volume_idx]    # [60, num_val_ids]

print(f"Volume target shape (train): {vol_train.shape}")
print(f"Volume target shape (val):   {vol_val.shape}")

# Save scalers for denormalization
val_means = val_out["means"][:, :, feature_indices]
val_stds  = val_out["stds"][:, :, feature_indices]

train_scalers = {
    "feat_means": train_means,
    "feat_stds": train_stds,
    "val_feat_means": val_means,
    "val_feat_stds": val_stds,
}

val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
print(f"Train instruments: {vol_train.shape[1]}")
print(f"Val instruments:   {vol_val.shape[1]}")

# Feature Engineering for XGBoost

XGBoost can't process raw time series directly. We extract **6 summary statistics** per feature
over the 600-timestep input window, producing a flat feature vector per instrument:

- **mean** — average level over the input window
- **std** — volatility / variation
- **min / max** — range extremes
- **last** — most recent value (closest to prediction window)
- **last60_mean** — average of the last minute of input (immediate context)

28 features x 6 stats = **168 features** per instrument.

In [ ]:
input_window = config.input_window  # 600

X_train_input = X_train_tensor[-input_window:, :, :]  # [600, N_train, 28]
X_val_input   = X_val_tensor[-input_window:, :, :]    # [600, N_val, 28]

def extract_summary_features(X):
    """Extract summary statistics from [T, N, F] tensor -> [N, F*6] numpy array."""
    feat_mean    = X.mean(dim=0)              # [N, F]
    feat_std     = X.std(dim=0)               # [N, F]
    feat_min     = X.min(dim=0).values        # [N, F]
    feat_max     = X.max(dim=0).values        # [N, F]
    feat_last    = X[-1, :, :]                # [N, F]
    feat_last60  = X[-60:, :, :].mean(dim=0)  # [N, F]

    stacked = torch.cat([feat_mean, feat_std, feat_min, feat_max, feat_last, feat_last60], dim=1)
    return stacked.cpu().numpy()

X_train_flat = extract_summary_features(X_train_input)  # [N_train, 168]
X_val_flat   = extract_summary_features(X_val_input)    # [N_val, 168]

# Replace any NaN/inf from zero-std features
X_train_flat = np.nan_to_num(X_train_flat, nan=0.0, posinf=0.0, neginf=0.0)
X_val_flat   = np.nan_to_num(X_val_flat, nan=0.0, posinf=0.0, neginf=0.0)

# Targets: [60, N] -> [N, 60]
y_train_flat = vol_train.T.cpu().numpy()
y_val_flat   = vol_val.T.cpu().numpy()

print(f"XGBoost feature matrix (train): {X_train_flat.shape}")
print(f"XGBoost feature matrix (val):   {X_val_flat.shape}")
print(f"XGBoost target (train):         {y_train_flat.shape}")
print(f"XGBoost target (val):           {y_val_flat.shape}")

stat_names = ['mean', 'std', 'min', 'max', 'last', 'last60_mean']
xgb_feature_names = []
for stat in stat_names:
    for col in config.feature_cols:
        xgb_feature_names.append(f'{col}_{stat}')
print(f"Total features: {len(xgb_feature_names)}")

# XGBoost Training

60 independent XGBoost regressors (one per second) via `MultiOutputRegressor`.
Each model predicts the z-scored log-volume for one specific second in the target window.

In [ ]:
from time import time

xgb_base = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=1,       # Single-threaded per model (outer wrapper parallelizes)
    verbosity=0,
)

# Train 60 independent XGBoost models (one per second)
model = MultiOutputRegressor(xgb_base, n_jobs=-1)

t0 = time()
model.fit(X_train_flat, y_train_flat)
train_time = time() - t0

val_preds = model.predict(X_val_flat)  # [N_val, 60]

print(f"Training time: {train_time:.1f}s (60 independent XGBoost models)")
print(f"Predictions shape: {val_preds.shape}")

In [ ]:
# Feature importance (averaged across the 60 per-second models)
importances = np.mean([est.feature_importances_ for est in model.estimators_], axis=0)

top_k = 15
top_idx = np.argsort(importances)[::-1][:top_k]

print(f'Top {top_k} features by importance (averaged across 60 models):')
print('-' * 55)
for rank, idx in enumerate(top_idx, 1):
    print(f'{rank:2d}. {xgb_feature_names[idx]:<40s} {importances[idx]:.4f}')

# Volume Prediction Quality

Evaluate R^2 and MSE. Denormalization path: z-score undo -> expm1 (undo log1p) -> raw volume.

Same evaluation as `mhf-volume.ipynb` for direct comparison with DeepLOB.

In [ ]:
all_preds = torch.tensor(val_preds, dtype=torch.float32)    # [N_val, 60]
all_targets = torch.tensor(y_val_flat, dtype=torch.float32)  # [N_val, 60]

# Denormalize: undo z-score -> undo log1p
vol_mean = train_scalers['val_feat_means'][:, :, volume_idx].squeeze(0).unsqueeze(1).cpu()
vol_std  = train_scalers['val_feat_stds'][:, :, volume_idx].squeeze(0).unsqueeze(1).cpu()

preds_log   = all_preds * vol_std + vol_mean
targets_log = all_targets * vol_std + vol_mean
preds_denorm   = torch.expm1(preds_log)
targets_denorm = torch.expm1(targets_log)

# Per-instrument R^2 (raw volume space)
y_mean = targets_denorm.mean(dim=1, keepdim=True)
ss_res = ((targets_denorm - preds_denorm) ** 2).sum(dim=1)
ss_tot = ((targets_denorm - y_mean) ** 2).sum(dim=1)
r2_per_inst = 1 - ss_res / ss_tot

valid_r2 = r2_per_inst[ss_tot > 1e-6]
final_r2 = valid_r2.mean().item()
final_mse = ((preds_denorm - targets_denorm) ** 2).mean().item()

# Normalized (z-score) space R^2
norm_y_mean = all_targets.mean(dim=1, keepdim=True)
norm_ss_res = ((all_targets - all_preds) ** 2).sum(dim=1)
norm_ss_tot = ((all_targets - norm_y_mean) ** 2).sum(dim=1)
norm_r2_per_inst = 1 - norm_ss_res / norm_ss_tot
norm_valid_r2 = norm_r2_per_inst[norm_ss_tot > 1e-6]
norm_r2 = norm_valid_r2.mean().item()

print('=' * 55)
print(f'VOLUME PREDICTION QUALITY - XGBoost ({data_asset})')
print('=' * 55)
print(f'Validation instruments: {len(val_ids)}')
print(f'Instruments with non-constant volume: {len(valid_r2)}')
print()
print('--- Raw volume space ---')
print(f'Denormalized MSE:    {final_mse:.6f}')
print(f'Denormalized R^2:    {final_r2:.6f}')
print(f'Median R^2 per inst: {valid_r2.median().item():.6f}')
print()
print('--- Normalized (z-score) space ---')
print(f'Normalized R^2:      {norm_r2:.6f}')
print(f'Median norm R^2:     {norm_valid_r2.median().item():.6f}')
print()
print(f'R^2 > 0 (any signal): {(valid_r2 > 0).float().mean().item()*100:.1f}%')
print(f'R^2 > 0.3 (decent):   {(valid_r2 > 0.3).float().mean().item()*100:.1f}%')
print(f'R^2 > 0.5 (good):     {(valid_r2 > 0.5).float().mean().item()*100:.1f}%')
print('=' * 55)
print()
print('For comparison, mid-price R^2 is typically 0.90-0.95.')
print('Literature reports volume R^2 of 0.2-0.6 at 15-MINUTE intervals.')

# Can Volume Predictions Improve Execution?

**VWAP-style approach:** Allocate more volume to seconds where predicted trading volume is high
(assumption: high volume = more liquidity = lower market impact).

In [ ]:
Y_val_raw = Y_raw[Y_raw['anonymized_id'].isin(val_ids)].copy()

vwap_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw['anonymized_id'] == anon_id].sort_values('time_in_hour')

    if len(df_inst) != 60:
        continue

    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]

    # Denormalize: undo z-score -> undo log1p -> raw volume
    vol_pred_raw = np.expm1(val_preds[i] * vol_std[i].item() + vol_mean[i].item())

    # VWAP-style: trade proportionally to predicted volume in last K seconds
    vol_window = vol_pred_raw[-K_SECONDS:]
    weights = np.maximum(vol_window, 0)

    if weights.sum() > 1e-8:
        weights = weights / weights.sum()
    else:
        weights = np.ones(K_SECONDS) / K_SECONDS

    positions = np.zeros(60)
    positions[-K_SECONDS:] = weights * volume_to_fill

    vol, avg_price = simulate_walk_the_book(
        positions, ask_prices, ask_vols, bid_prices, bid_vols
    )

    if vol > 0 and not np.isnan(avg_price):
        impl_error = np.abs(avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / vol)
        vwap_bps.append(impl_error * vol_penalty)

vwap_bps = np.array(vwap_bps)

print()
print('=' * 55)
print('VOLUME-WEIGHTED (VWAP-STYLE) EXECUTION')
print('=' * 55)
print(f'Window: last {K_SECONDS}s, weights proportional to predicted volume')
print(f'Instruments evaluated: {len(vwap_bps)}')
print(f'Mean:   {vwap_bps.mean():.4f} bps')
print(f'Median: {np.median(vwap_bps):.4f} bps')
print(f'Std:    {vwap_bps.std():.4f} bps')
print('=' * 55)

In [ ]:
# --- Blended Volume-Weighted + TWAP (10% model, 90% TWAP) ---

blended_bps = []
ALPHA = 0.1

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw['anonymized_id'] == anon_id].sort_values('time_in_hour')

    if len(df_inst) != 60:
        continue

    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]

    vol_pred_raw = np.expm1(val_preds[i] * vol_std[i].item() + vol_mean[i].item())

    vol_window = vol_pred_raw[-K_SECONDS:]
    weights = np.maximum(vol_window, 0)
    if weights.sum() > 1e-8:
        weights = weights / weights.sum()
    else:
        weights = np.ones(K_SECONDS) / K_SECONDS

    model_positions = np.zeros(60)
    model_positions[-K_SECONDS:] = weights * volume_to_fill

    twap_positions = np.zeros(60)
    twap_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS

    blended_positions = ALPHA * model_positions + (1 - ALPHA) * twap_positions
    blended_positions = blended_positions * (volume_to_fill / blended_positions.sum())

    vol, avg_price = simulate_walk_the_book(
        blended_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )

    if vol > 0 and not np.isnan(avg_price):
        impl_error = np.abs(avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / vol)
        blended_bps.append(impl_error * vol_penalty)

blended_bps = np.array(blended_bps)

pct = int(ALPHA * 100)
print()
print('=' * 55)
print(f'BLENDED VOLUME-WEIGHTED ({pct}% model / {100-pct}% TWAP)')
print('=' * 55)
print(f'Instruments evaluated: {len(blended_bps)}')
print(f'Mean:   {blended_bps.mean():.4f} bps')
print(f'Median: {np.median(blended_bps):.4f} bps')
print('=' * 55)

# TWAP Baseline

In [ ]:
baseline_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw['anonymized_id'] == anon_id].sort_values('time_in_hour')

    if len(df_inst) != 60:
        continue

    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]

    baseline_positions = np.zeros(60)
    baseline_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS

    baseline_vol, baseline_avg_price = simulate_walk_the_book(
        baseline_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )

    if baseline_vol > 0 and not np.isnan(baseline_avg_price):
        impl_error = np.abs(baseline_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / baseline_vol)
        baseline_bps.append(impl_error * vol_penalty)

baseline_bps = np.array(baseline_bps)

print()
print('=' * 55)
print(f'BASELINE (TWAP-{K_SECONDS}s) IMPLEMENTATION ERROR')
print('=' * 55)
print(f'Instruments evaluated: {len(baseline_bps)}')
print(f'Mean:   {baseline_bps.mean():.4f} bps')
print(f'Median: {np.median(baseline_bps):.4f} bps')
print(f'Std:    {baseline_bps.std():.4f} bps')
print('=' * 55)

# Summary Comparison

In [ ]:
print('=' * 65)
print(f'VOLUME PREDICTION EXPERIMENT SUMMARY - XGBoost ({data_asset})')
print('=' * 65)
print()
print('--- Model ---')
print(f'XGBoost with {len(xgb_feature_names)} summary features')
print('60 independent models (one per second via MultiOutputRegressor)')
print(f'Training time: {train_time:.1f}s')
print()
print('--- Prediction Quality ---')
print(f'Normalized R^2:     {norm_r2:.4f}')
print(f'Denormalized R^2:   {final_r2:.4f}')
print(f'Denormalized MSE:  {final_mse:.4f}')
print('(DeepLOB VolumeModel R^2 on same data: ~-2.73)')
print()
print('--- Execution Quality (BPS, lower is better) ---')
twap_label = f'TWAP-{K_SECONDS}s (baseline)'
vwap_label = 'Volume-weighted (VWAP-style)'
pct = int(ALPHA * 100)
blend_label = f'Blended ({pct}% vol / {100-pct}% TWAP)'
print(f'{"Strategy":<40} {"Mean BPS":>10} {"Median BPS":>12}')
print('-' * 62)
print(f'{twap_label:<40} {baseline_bps.mean():>10.4f} {np.median(baseline_bps):>12.4f}')
print(f'{vwap_label:<40} {vwap_bps.mean():>10.4f} {np.median(vwap_bps):>12.4f}')
print(f'{blend_label:<40} {blended_bps.mean():>10.4f} {np.median(blended_bps):>12.4f}')
print()
print('--- Delta vs TWAP ---')
print(f'Volume-weighted vs TWAP: {vwap_bps.mean() - baseline_bps.mean():+.4f} bps')
print(f'Blended vs TWAP:        {blended_bps.mean() - baseline_bps.mean():+.4f} bps')
print('=' * 65)

# Conclusions

XGBoost with hand-crafted summary features produces similar results to DeepLOB on volume prediction.

**Key finding:** Both deep learning (DeepLOB) and gradient boosting (XGBoost) fail to predict per-second
volume with positive R^2. This confirms that the difficulty is **not** model architecture — it's the
fundamental unpredictability of per-second trading volume.

This matches the literature: DeepLOB^v achieves R^2 = 0.624 on equities at 15-minute intervals, while
XGBoost matches it at R^2 = 0.622. At per-second crypto granularity, neither approach can find signal.

**Conclusion:** Volume prediction at per-second granularity does not improve execution quality, regardless
of model class. Focus on mid-price and spread prediction where the oracle ceiling is higher.